# HeatShield AI
## Feature Analysis & Correlation Study

Objectif :
Analyser les variables créées par le feature engineering
et comprendre lesquelles influencent le plus le risque d'incendie.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8")

In [ ]:
DATA_PATH = "../data/processed/algeria_weather_features_v1.csv"
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
print("Colonnes:", df.columns.tolist())

In [ ]:
feature_cols = [
    "temp_max", "temp_mean", "humidity", "wind_max", "rain",
    "et0", "vpd_max", "radiation",
    "heat_humidity_ratio", "wind_drought_index", "dry_air_index",
    "rain_protection",
    "temp_3d_avg", "temp_7d_avg",
    "humidity_3d_avg", "humidity_7d_avg",
    "wind_3d_avg", "wind_7d_avg",
    "rain_3d_sum", "rain_7d_sum",
    "et0_7d_sum", "vpd_7d_avg",
    "hot_day_38", "hot_day_40", "hot_day_42",
    "hot_days_38_rolling", "hot_days_40_rolling",
    "cumulative_heat_stress", "heatwave_intensity",
    "consecutive_hot_days_38",
    "risk_score"
]

existing_cols = [c for c in feature_cols if c in df.columns]
df_corr = df[existing_cols].copy()
df_corr.head()

In [ ]:
corr = df_corr.corr(numeric_only=True)
corr["risk_score"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(16,12))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.show()

In [ ]:
top_corr = corr["risk_score"].drop("risk_score").sort_values(key=abs, ascending=False).head(15)
top_corr

plt.figure(figsize=(10,6))
top_corr.plot(kind="barh")
plt.title("Top 15 variables les plus corrélées au risk_score")
plt.xlabel("Corrélation")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="risk_level", y="heatwave_intensity",
            order=["Faible","Modéré","Élevé","Critique"])
plt.title("Heatwave intensity selon le niveau de risque")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="risk_level", y="consecutive_hot_days_38",
            order=["Faible","Modéré","Élevé","Critique"])
plt.title("Jours chauds consécutifs selon le niveau de risque")
plt.show()

In [ ]:
plt.figure(figsize=(9,6))
sns.scatterplot(
    data=df.sample(min(6000, len(df)), random_state=42),
    x="heat_humidity_ratio", y="risk_score", hue="risk_level",
    hue_order=["Faible","Modéré","Élevé","Critique"],
    alpha=0.6
)
plt.title("Heat/Humidity Ratio vs Risk Score")
plt.show()

In [ ]:
sample_wilaya = "Alger"
sub = df[df["wilaya_nom"] == sample_wilaya].sort_values("date")

plt.figure(figsize=(12,5))
plt.plot(pd.to_datetime(sub["date"]), sub["temp_max"], label="Temp max")
plt.plot(pd.to_datetime(sub["date"]), sub["temp_7d_avg"], label="Temp 7d avg")
plt.title(f"Température au fil du temps — {sample_wilaya}")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(pd.to_datetime(sub["date"]), sub["risk_score"])
plt.title(f"Évolution du risk score — {sample_wilaya}")
plt.ylabel("Risk score")
plt.show()

In [ ]:
corr.to_csv("../data/processed/correlation_matrix_features.csv")
top_corr.to_csv("../data/processed/top_correlations_with_risk.csv")
print("Fichiers sauvegardés.")